# Track 00 — 부트스트랩

**구성** : 각 `Session`은 코드 설명(텍스트) -> 코드 -> 해석(텍스트) 순으로 정리되어 있습니다.

**목표** : Track 00의 목표는 저장소·`.env` 환경을 점검하고 EXAONE 첫 API 호출(기준·thinking·streaming)을 비교하는 것입니다.

**산출물:** `_out/first_calls.json`


## 설치 (저장소 루트, 한 번)

```bash
cd /path/to/<cookbook-root>
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt && pip install -e ./exaone
cp .env.example .env
```


## Session 1. 환경 점검

Python · 패키지 · `.env` · API 키를 **단계별**로 확인합니다.


### Session 1-1. exaone import

**하는 일:** `exaone` 패키지를 import 하고 저장소 루트를 출력합니다.

**정상:** `ROOT`, `exaone` 버전, `executable` 이 보임

**의미:** exaone/ 코어의 설치와 커널 경로가 맞는지 먼저 봅니다.



In [ ]:
import importlib
import sys
from pathlib import Path

# (en) Requires editable install at repo root.
# (kr) 저장소 루트에서 editable 설치 필요.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone 가 설치되지 않았습니다. pip install -r requirements.txt && pip install -e ./exaone"
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
print("ROOT       :", ROOT)
print("exaone     :", exaone.__version__)
print("Python     :", sys.version.split()[0])
print("executable :", sys.executable)

**출력 해석:** `ROOT`·`exaone` 버전·`executable` 이 보이면 editable 설치와 커널 경로가 맞습니다.



### Session 1-2. 패키지 점검

**하는 일:** 필수 패키지가 import 되는지 확인합니다.

**정상:** 표에 `OK` 만 보이거나, `MISSING` 이 없음

**의미:** API 호출 전에 의존성 누락을 잡습니다.



In [ ]:
from importlib.metadata import PackageNotFoundError, version

# (en) Record which dependencies import cleanly.
# (kr) 어떤 의존성이 정상 import 되는지 기록한다.
REQUIRED = ["openai", "requests", "dotenv", "jsonschema", "exaone"]
packages: dict[str, str] = {}
for name in REQUIRED:
    try:
        if name == "exaone":
            packages[name] = f"OK ({exaone.__version__})"
        elif name == "dotenv":
            importlib.import_module("dotenv")
            packages[name] = "OK"
        else:
            packages[name] = f"OK ({version(name)})"
    except (ModuleNotFoundError, PackageNotFoundError) as exc:
        packages[name] = f"MISSING - {type(exc).__name__}"

print("[패키지]")
for name, state in packages.items():
    print(f"  {name:<10} {state}")
missing = [n for n, s in packages.items() if s.startswith("MISSING")]

**출력 해석:** 표에 `OK` 만 있으면 의존성이 갖춰진 상태입니다. `MISSING` 이 보이면 해당 패키지를 다시 설치하세요.



### Session 1-3. `.env` 파일

**하는 일:** 루트 `.env` 가 있는지 보고, 없으면 예시에서 복사합니다.

**정상:** `.env 있음` 또는 `.env 생성됨` 메시지

**의미:** exaone 은 저장소 루트 `.env` 한 파일만 읽습니다.



In [ ]:
import shutil

# (en) Create .env from example when missing.
# (kr) 없으면 .env.example 에서 .env 를 만든다.
env_example = ROOT / ".env.example"
env_file = ROOT / ".env"
if env_file.exists():
    print(".env 있음:", env_file)
elif env_example.is_file():
    shutil.copy(env_example, env_file)
    print(".env 생성됨 — 세 키를 채우세요.")
else:
    print(".env.example 이 없습니다.")
loaded_from = exaone.load_project_env()
print("load_project_env():", loaded_from)

**출력 해석:** `.env 있음` 또는 `.env 생성됨` 이면 환경 파일이 준비됐습니다. 키 **값**은 출력되지 않습니다.



### Session 1-4. API 키·SSL

**하는 일:** 필수 환경 변수 존재 여부와 SSL 옵션을 출력합니다.
**정상:** `EXAONE_API_KEY` 가 `OK (length=...)`
**의미:** 키가 있어야 Session 2 API 호출이 됩니다.



In [ ]:
import os

# (en) Never print secret values — only presence and length.
# (kr) 비밀값은 출력하지 않고 있다/없다·길이만 표시한다.
env_keys = {}
for name in ("EXAONE_API_KEY", "EXAONE_BASE_URL", "EXAONE_MODEL"):
    value = os.environ.get(name, "").strip()
    env_keys[name] = f"OK (length={len(value)})" if value else "MISSING"
print("[필수 키]")
for name, state in env_keys.items():
    print(f"  {name:<18} {state}")

ssl_flags = {
    "REQUESTS_CA_BUNDLE": bool(os.environ.get("REQUESTS_CA_BUNDLE", "").strip()),
    "SSL_CERT_FILE": bool(os.environ.get("SSL_CERT_FILE", "").strip()),
    "DISABLE_SSL_VERIFY": exaone.config.get_disable_ssl_verify(),
}
print("\n[SSL 옵션]")
for name, on in ssl_flags.items():
    print(f"  {'ON ' if on else 'off'}  {name}")

HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
print("\nHAS_API =", HAS_API)

**출력 해석:** `EXAONE_API_KEY` 가 `OK (length=…)` 이고 `HAS_API = True` 이면 Session 2 API 호출을 진행할 수 있습니다.

**이상:** `MISSING` 이면 `.env` 에 키를 채운 뒤 커널을 재시작하세요.



## Session 2. 첫 API 호출

기본 질문에 **knob 하나씩**만 바꿔 가며 체감합니다. thinking 은 효용이 드러나도록 **추론형 질문**으로 OFF↔ON 을 비교합니다.

| 단계 | 질문 | 바꾸는 것 | 체감 |
|---|---|---|---|
| Session 2-2 | 소개 | 없음 | 답이 한 번에 출력 |
| Session 2-3 | 추론(사탕 개수) | thinking OFF→ON | OFF는 `reasoning` 채널 없음, ON은 사고 과정이 `reasoning` 으로 분리 |
| Session 2-4 | 소개 | streaming 켜기 | 글자가 흘러나옴 |

### Session 2-1. 질문·클라이언트

**하는 일:** 질문 리스트와 `client` 를 준비합니다.

**입력:** `user` 메시지 1줄

**정상:** `client` 생성 후 에러 없음

**의미:** 이후 A/B/C 셀이 같은 `messages`·`client` 를 재사용합니다.



In [ ]:
# (en) feel free to change the prompt
# (kr) 자유롭게 질문을 바꿔보세요.
prompt = "EXAONE을 한 문장으로 소개해줘."
messages = [exaone.llm.ExaoneMessage(role="user", content=prompt)]
client = exaone.integrations.build_llm_from_env()
print("질문:", messages[0].content)
print("model:", client.model)

**출력 해석:** `질문:` 에 대해서 `model:`이 답변할 준비를 했습니다. 아직 Tool이나 Skill이 추가되지 않아서, 간단한 질의응답만 가능합니다.

### Session 2-2. 기준 chat()

**하는 일:** 옵션 없이 `chat()` 1회 (streaming 미사용)

**정상:** `A:` 뒤 한글 답

**의미:** 가장 기본적인 동기 응답입니다. (`chat()` 기본값은 `enable_thinking=True` — 켜고 끄는 비교는 2-3 에서 합니다.)


In [ ]:
a = client.chat(messages)
print("A:", a.content)

**출력 해석:** `A:` 뒤에 한글 답이 있으면 기본 동기 응답이 정상입니다. 비어 있으면 API·모델 설정을 확인하세요.



### Session 2-3. thinking

**하는 일:** 같은 **추론형 질문**(다단계 계산)을 `enable_thinking` **OFF → ON** 으로 보내 비교합니다. `chat()` 기본값이 `True` 라, OFF 는 `enable_thinking=False` 로 **명시해서** 꺼야 합니다.

**정상:** `[thinking OFF]` 는 `reasoning` 채널 **없음**, `[thinking ON]` 은 `reasoning` 채널 **있음**(단계별 사고 과정)

**의미:** thinking 은 모델이 답하기 전에 단계적으로 추론하고, 그 과정을 `reasoning` 채널로 분리합니다. 풀이를 직접 읽고 검증할 수 있어, 수학·데이터분석처럼 **다단계 추론**이 필요한 문제일수록 유용합니다.


In [ ]:
# (en) thinking shines on multi-step problems, so use a reasoning question.
# (kr) thinking 은 여러 단계를 거치는 문제에서 빛나므로, 추론형 질문을 쓴다.
reasoning_q = [
    exaone.llm.ExaoneMessage(
        role="user",
        content=(
            "한 상자에 사탕이 24개 있습니다. 첫째 날 전체의 1/4을 먹고, "
            "둘째 날 '남은 것'의 1/3을 먹었습니다. 남은 사탕은 몇 개인가요? "
            "마지막 줄에 '정답: N개' 형식으로 답해줘."
        ),
    )
]

# (en) chat() defaults to enable_thinking=True, so turn it OFF explicitly here.
# (kr) chat() 기본값이 enable_thinking=True 이므로, 여기서는 명시적으로 끈다.
b_off = client.chat(
    reasoning_q, options=exaone.llm.ExaoneGenerateOptions(enable_thinking=False)
)
print("[thinking OFF] reasoning 채널:", "있음" if b_off.reasoning_content else "없음")
print(b_off.content, "\n")

# (en) thinking ON — the step-by-step thought lands in a separate reasoning channel.
# (kr) thinking 켬 — 단계별 사고 과정이 별도 reasoning 채널에 담긴다.
b = client.chat(
    reasoning_q, options=exaone.llm.ExaoneGenerateOptions(enable_thinking=True)
)
print("[thinking ON] reasoning 채널:", "있음" if b.reasoning_content else "없음")
print("[thinking ON] 최종 답:", b.content)
if b.reasoning_content:
    print("[thinking ON] reasoning(앞부분):", b.reasoning_content[:300])

**출력 해석:** 정답은 12개입니다(24 → 18 → 12). 이 문제는 OFF/ON 모두 맞히지만, **차이는 풀이가 담기는 위치**입니다 — OFF 는 `reasoning` 채널이 없어 풀이가 답에 섞이고, ON 은 사고 과정을 `reasoning` 채널로 분리해 검증할 수 있게 합니다. 문제가 어려워질수록 이 '선(先)추론'이 안정성을 높입니다.


### Session 2-4. streaming

**하는 일:** `chat_stream()` 으로 모델이 생성하는 답변을 실시간으로 출력합니다.

**정상:** `C:` 뒤 글자가 이어서 출력

**의미:** 모델과 실시간으로 소통하는 경험을 합니다.


In [ ]:
print("C:", end=" ")
chunks = []
for ch in client.chat_stream(messages):
    chunks.append(ch)
    if ch.kind == "text":
        print(ch.text, end="", flush=True)
print()
c = exaone.llm.stream_chunks_to_response(iter(chunks))

**출력 해석:** `C:` 뒤 글자가 이어서 나오면 스트리밍이 정상입니다. 



### Session 2-5. 저장

**하는 일:** 앞서 실행한 A/B/C 결과를 `first_calls.json` 에 저장

**정상:** `saved:` 경로 출력

**의미:** AI 모델과 소통한 첫 결과물입니다. Track 01 이후에도 참고하니 꼭 저장해주세요.



In [ ]:
import json

Path("_out").mkdir(exist_ok=True)
out = Path("_out") / "first_calls.json"
out.write_text(
    json.dumps(
        {
            "A": {"content": a.content},
            "B": {"content": b.content, "reasoning": b.reasoning_content},
            "C": {"content": c.content},
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("saved:", out.resolve())

**출력 해석:** `saved:` 경로가 보이면 `_out/first_calls.json` 에 A/B/C 결과가 저장됐습니다.



## 체크포인트

- [ ] Session 1-4 `HAS_API = True` 문구 나오는 것 확인
- [ ] Session 2-2~2-4 `A:` / `B:` / `C:` 로 AI의 응답을 출력해보기
- [ ] Session 2-5 `_out/first_calls.json`로 AI의 응답을 저장해보기

**다음:** Track 01 — EXAONE Foundation